# ARCHS4 CLAMP models with all pathways prior at 1% sample coverage (Fixed K from hall 100%)

**Environment:** `clamp-analyses`

Self-contained pipeline at 1% sample coverage using K fixed from the 100% hall model.

**Key difference from `06_bp_coverage_rshall`:** CLAMP_K is loaded from `hall_coverage_rs100_seed_1` instead of being inferred at this coverage level, so the number of LVs matches the full dataset model. This tests the effect of K on model quality at lower coverage.

Steps (repeated for each seed):
1. Subsample ARCHS4 data to 1% coverage
2. Compute SVD
3. Load CLAMP_K from `hall_coverage_rs100_seed_1`
4. Run CLAMPbase with fixed K
5. Run CLAMPfull with all pathways prior
6. Save results to `hall_fixedK_coverage_rs1_seed_*`

## Load libraries

In [ ]:
library(bigstatsr)
library(data.table)
library(dplyr)
library(rsvd)
library(Matrix)
library(here)
library(CLAMP)
library(PCAtools)

source(here("config.R"))

## Configuration

In [ ]:
base_output_dir <- config$ARCHS4$DATASET_FOLDER

coverage     <- 0.01
coverage_pct <- coverage * 100

pathways_path <- here::here('data/pathways')

N_CORES    <- config$ARCHS4$CLAMP_PARAMS$RANDOM_SVD_N_CORES
MULTIPLIER <- 100
MAX_ITER   <- 5000

base_seed <- config$ARCHS4$CLAMP_PARAMS$RANDOM_SVD_SEED
seeds     <- base_seed + as.integer(coverage_pct) * 10L + 0:2
n_runs    <- length(seeds)
message("Coverage: ", coverage_pct, "% — will run ", n_runs, " seeds (fixed K from hall 100%)")

## Load preprocessed data and pathways

In [ ]:
meta            <- readRDS(file.path(base_output_dir, "metadata_filtered.rds"))
n_genes_thin    <- meta$n_genes_thin
n_samples_total <- meta$n_samples
archs4_genes    <- meta$gene_symbols_thin

all_samples        <- readRDS(file.path(base_output_dir, "all_samples.rds"))
sample_names_total <- all_samples[seq_len(n_samples_total)]

hall_gmt     <- CLAMP:::read_gmt(file.path(pathways_path, "h.all.v2026.1.Hs.symbols.gmt"))
reactome_gmt <- CLAMP:::read_gmt(file.path(pathways_path, "c2.cp.reactome.v2026.1.Hs.symbols.gmt"))
gocc_gmt     <- CLAMP:::read_gmt(file.path(pathways_path, "c5.go.cc.v2026.1.Hs.symbols.gmt"))
c8_gmt       <- CLAMP:::read_gmt(file.path(pathways_path, "c8.all.v2026.1.Hs.symbols.gmt"))

names(hall_gmt)     <- paste0("HALL_",     names(hall_gmt))
names(reactome_gmt) <- paste0("REACTOME_", names(reactome_gmt))
names(gocc_gmt)     <- paste0("GOCC_",     names(gocc_gmt))
names(c8_gmt)       <- paste0("C8_",       names(c8_gmt))

all_pathways_list <- list(
  HALL     = hall_gmt,
  REACTOME = reactome_gmt,
  GOCC     = gocc_gmt,
  C8       = c8_gmt
)

all_pathways_pathMat <- gmtListToSparseMat(all_pathways_list)
all_pathways_matched <- getMatchedPathwayMat(all_pathways_pathMat, archs4_genes)
message("Loaded and matched all pathways matrix")

archs4_fbm_filt <- FBM(
  nrow        = n_genes_thin,
  ncol        = n_samples_total,
  backingfile = file.path(base_output_dir, "fbm_filtered"),
  create_bk   = FALSE
)
message("Loaded FBM with ", n_genes_thin, " genes and ", n_samples_total, " samples")

## Run 1% coverage models with fixed K (SVD → CLAMPbase (fixedK) → CLAMPfull)

In [ ]:
output_dir <- file.path(base_output_dir, "07_bp_coverage_hall_fixedK")
dir.create(output_dir, showWarnings = FALSE, recursive = TRUE)

k100_dir <- file.path(base_output_dir, "hall_coverage_rs100_seed_1")
CLAMP_K  <- readRDS(file.path(k100_dir, "CLAMP_K.rds"))
message("Using fixed CLAMP_K = ", CLAMP_K, " (from hall_coverage_rs100_seed_1)")

n_samples_target <- round(n_samples_total * coverage)
message("Target samples per run: ", n_samples_target, " (", coverage_pct, "% of ", n_samples_total, ")")

for (run_idx in seq_len(n_runs)) {
  current_seed <- seeds[run_idx]
  message("\n", strrep("=", 60))
  message("RUN ", run_idx, "/", n_runs, " - Seed: ", current_seed)
  message(strrep("=", 60))

  dst_dir <- file.path(output_dir,
    paste0("hall_fixedK_coverage_rs", coverage_pct, "_seed_", run_idx))
  dir.create(dst_dir, showWarnings = FALSE, recursive = TRUE)

  set.seed(current_seed)
  sample_idx   <- sort(sample(seq_len(n_samples_total), n_samples_target))
  n_samples    <- length(sample_idx)
  sample_names <- sample_names_total[sample_idx]
  message("Randomly selected ", n_samples, " samples")

  saveRDS(list(
    run = run_idx, seed = current_seed, coverage = coverage,
    n_samples = n_samples, sample_idx = sample_idx, sample_names = sample_names,
    sampling_method = "random_sampling"
  ), file = file.path(dst_dir, "subsample_info.rds"))

  message("Creating subsampled FBM...")
  Y_sub <- big_copy(
    archs4_fbm_filt,
    ind.col     = sample_idx,
    backingfile = file.path(dst_dir, "fbm_subsampled")
  )

  message("Computing SVD...")
  SVD_K <- max(CLAMP_K, round(min(n_samples - 1, n_genes_thin - 1) / 4))

  set.seed(current_seed)
  if (N_CORES > 1) {
    options(bigstatsr.check.parallel.blas = FALSE)
    blas_nproc <- getOption("default.nproc.blas")
    options(default.nproc.blas = NULL)
  }

  svd_result <- big_randomSVD(Y_sub, k = SVD_K, ncores = N_CORES)

  if (N_CORES > 1) {
    options(bigstatsr.check.parallel.blas = TRUE)
    options(default.nproc.blas = blas_nproc)
  }

  valid_idx    <- which(!is.nan(svd_result$d))
  svd_result$d <- svd_result$d[valid_idx]
  svd_result$u <- svd_result$u[, valid_idx, drop = FALSE]
  svd_result$v <- svd_result$v[, valid_idx, drop = FALSE]
  saveRDS(svd_result, file = file.path(dst_dir, "svd.rds"))

  saveRDS(CLAMP_K, file = file.path(dst_dir, "CLAMP_K.rds"))

  message("Running CLAMPbase with fixed K...")
  baseRes <- CLAMPbase(Y = Y_sub, svdres = svd_result, trace = TRUE, clamp_k = CLAMP_K)
  baseRes$Z <- data.frame(baseRes$Z)
  rownames(baseRes$Z) <- archs4_genes
  baseRes$B <- data.frame(baseRes$B)
  colnames(baseRes$B) <- sample_names
  saveRDS(baseRes, file = file.path(dst_dir, "CLAMPbase.rds"))

  model_dir <- file.path(dst_dir, "CLAMPbase")
  dir.create(model_dir, showWarnings = FALSE, recursive = TRUE)
  write.csv(baseRes$B, file.path(model_dir, "B.csv"))
  write.csv(baseRes$Z, file.path(model_dir, "Z.csv"))

  message("Running CLAMPfull with hall prior (fixed K)...")
  fullRes <- CLAMPfull(
    Y                 = Y_sub,
    svdres            = svd_result,
    priorMat          = all_pathways_matched,
    clamp.base.result = baseRes,
    use_cpp           = TRUE,
    trace             = TRUE,
    multiplier        = MULTIPLIER,
    max.iter          = MAX_ITER,
    clamp_k           = CLAMP_K
  )
  fullRes$Z <- data.frame(fullRes$Z)
  rownames(fullRes$Z) <- archs4_genes
  fullRes$B <- data.frame(fullRes$B)
  colnames(fullRes$B) <- sample_names
  fullRes$summary <- fullRes$summary %>%
    dplyr::rename(LV = LV_index) %>%
    dplyr::mutate(LV = paste0('LV', LV))

  saveRDS(fullRes, file = file.path(dst_dir, "CLAMPfull_hall.rds"))
  model_dir <- file.path(dst_dir, "CLAMPfull_hall")
  dir.create(model_dir, showWarnings = FALSE, recursive = TRUE)
  write.csv(fullRes$B,       file.path(model_dir, "B.csv"))
  write.csv(fullRes$Z,       file.path(model_dir, "Z.csv"))
  write.csv(fullRes$summary, file.path(model_dir, "summary.csv"))

  rm(Y_sub, svd_result, baseRes, fullRes)
  gc()
}

message("\n", strrep("=", 60))
message("All ", n_runs, " runs completed!")
message(strrep("=", 60))